# Whisper Fine-tuning Lab (Zeroth **or** Recorded Data)
This notebook supports **two data sources**:

- **Zeroth (HF Dataset saved_to_disk)**: you already prepared it as `.../hf_dataset_zeroth_whisper_small/{train,eval}`
- **Recorded data** collected in the notebook recorder UI: `~/data_team/audio/...` + `~/data_team/refs.csv`

You can switch modes via `DATA_MODE` below.

> Output: LoRA adapter + evaluation (WER/CER) before/after.

In [1]:
# If needed (run once)
!pip install -U "transformers>=4.40" "datasets>=2.18" accelerate peft jiwer soundfile librosa

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os
from pathlib import Path

# -----------------------------
# 1) Choose data mode
# -----------------------------
# "zeroth"   : load_from_disk(BASE_OUT / "hf_dataset_zeroth_whisper_small"/{train,eval})
# "recorded" : build dataset from ~/data_<mode>/refs.csv and ~/data_<mode>/audio
DATA_MODE = "zeroth"   # <- change to "recorded" if using your recorded dataset

HOME = Path.home()

# Zeroth HF dataset root (already prepared by you)
BASE_OUT = HOME / "data_zeroth_hf"
ZEROTH_DS_ROOT = BASE_OUT / "hf_dataset_zeroth_whisper_small"  # contains train/ eval/

ZEROTH_ROOT = HOME / "data_zeroth"
ZEROTH_REFS = ZEROTH_ROOT / "refs.csv"
ZEROTH_TRAIN_AUDIO = ZEROTH_ROOT / "train_data_01"
ZEROTH_TEST_AUDIO  = ZEROTH_ROOT / "test_data_01"

# Recorded dataset root (DATA_MODE가 recorded일 때만 의미)
REC_ROOT = HOME / f"data_{DATA_MODE}"   # ex) ~/data_recorded
REC_AUDIO_DIR = REC_ROOT / "audio"
REC_REFS_CSV  = REC_ROOT / "refs.csv"

# Output directory
OUT_DIR = HOME / "outputs_asr" / f"whisper_lora_{DATA_MODE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_MODE:", DATA_MODE)
print("OUT_DIR  :", OUT_DIR)

print("ZEROTH_ROOT exists:", ZEROTH_ROOT.exists(), ZEROTH_ROOT)
print("ZEROTH_DS_ROOT exists:", ZEROTH_DS_ROOT.exists(), ZEROTH_DS_ROOT)

DATA_MODE: zeroth
OUT_DIR  : /home/data/bootcamp/user2/outputs_asr/whisper_lora_zeroth
ZEROTH_ROOT exists: True /home/data/bootcamp/user2/data_zeroth
ZEROTH_DS_ROOT exists: True /home/data/bootcamp/user2/data_zeroth_hf/hf_dataset_zeroth_whisper_small


## (Optional) Quick sanity checks
- If `DATA_MODE="zeroth"`, ensure these exist:
  - `ZEROTH_DS_ROOT/train`
  - `ZEROTH_DS_ROOT/eval`
- If `DATA_MODE="recorded"`, ensure these exist:
  - `REC_REFS_CSV`
  - `REC_AUDIO_DIR/<studentXX>/*.wav`

In [3]:
if DATA_MODE == "zeroth":
    print("Zeroth train exists:", (ZEROTH_DS_ROOT / "train").exists())
    print("Zeroth eval  exists:", (ZEROTH_DS_ROOT / "eval").exists())
elif DATA_MODE == "recorded":
    print("refs.csv exists:", REC_REFS_CSV.exists(), REC_REFS_CSV)
    print("audio dir exists:", REC_AUDIO_DIR.exists(), REC_AUDIO_DIR)
else:
    raise ValueError("DATA_MODE must be 'zeroth' or 'recorded'")

Zeroth train exists: True
Zeroth eval  exists: True


# 2) Load dataset
We will normalize columns to:
- `audio` : `datasets.Audio`
- `text`  : transcript string

In [4]:
import pandas as pd
from datasets import load_from_disk, Dataset, Audio

def load_zeroth_dataset():
    ds_train = load_from_disk(str(ZEROTH_DS_ROOT / "train"))
    ds_eval  = load_from_disk(str(ZEROTH_DS_ROOT / "eval"))
    return ds_train, ds_eval

def load_recorded_dataset(eval_ratio: float = 0.1, seed: int = 42):
    if not REC_REFS_CSV.exists():
        raise FileNotFoundError(f"refs.csv not found: {REC_REFS_CSV}")

    df = pd.read_csv(REC_REFS_CSV)

    # expected columns: file, transcript
    if "file" not in df.columns or "transcript" not in df.columns:
        raise ValueError("refs.csv must have columns: file, transcript")

    df = df.copy()
    df["file"] = df["file"].astype(str).str.strip()
    df["transcript"] = df["transcript"].astype(str).fillna("").str.strip()
    df = df[df["file"].str.len() > 0]
    df = df[df["transcript"].str.len() > 0]

    # build absolute wav paths
    df["path"] = df["file"].apply(lambda p: str((REC_AUDIO_DIR / p).resolve()))

    # keep only existing
    df = df[df["path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

    if len(df) < 2:
        raise ValueError("Not enough recorded samples. Record more rows and fill transcripts.")

    ds = Dataset.from_pandas(df[["path", "transcript"]], preserve_index=False)
    ds = ds.rename_columns({"transcript": "text"})
    ds = ds.cast_column("path", Audio(sampling_rate=16000))
    ds = ds.rename_columns({"path": "audio"})

    split = ds.train_test_split(test_size=eval_ratio, seed=seed)
    return split["train"], split["test"]

if DATA_MODE == "zeroth":
    ds_train, ds_eval = load_zeroth_dataset()
elif DATA_MODE == "recorded":
    ds_train, ds_eval = load_recorded_dataset()
else:
    raise ValueError("DATA_MODE must be 'zeroth' or 'recorded'")

print(ds_train)
print(ds_eval)
print("train columns:", ds_train.column_names)
print("eval  columns:", ds_eval.column_names)

Loading dataset from disk:   0%|          | 0/41 [00:00<?, ?it/s]

Dataset({
    features: ['input_features', 'labels'],
    num_rows: 21148
})
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 1114
})
train columns: ['input_features', 'labels']
eval  columns: ['input_features', 'labels']


In [5]:
from datasets import Audio

# Normalize column names to: audio, text
def normalize(ds, text_candidates=("text", "transcript", "sentence", "transcription")):
    print("columns:", ds.column_names)
    cols = set(ds.column_names)

    # Case A) already-prepared for Whisper Trainer
    if {"input_features", "labels"}.issubset(cols):
        print("Detected preprocessed Whisper dataset (input_features/labels). No normalization needed.")
        return ds

    # Case B) raw dataset (needs audio/text)
    audio_col = None
    if "audio" in cols:
        audio_col = "audio"
    else:
        for c in ds.column_names:
            if "audio" in c.lower() or "wav" in c.lower():
                audio_col = c
                break
    if audio_col is None:
        raise ValueError(f"Cannot find audio column. columns={ds.column_names}")

    text_col = None
    for cand in text_candidates:
        if cand in cols:
            text_col = cand
            break
    if text_col is None:
        raise ValueError(f"Cannot find text column. Tried {text_candidates}. columns={ds.column_names}")

    rename_map = {}
    if audio_col != "audio":
        rename_map[audio_col] = "audio"
    if text_col != "text":
        rename_map[text_col] = "text"
    if rename_map:
        ds = ds.rename_columns(rename_map)

    try:
        ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    except Exception as e:
        print("cast_column(audio) skipped:", e)

    print("Normalized columns:", ds.column_names)
    return ds

ds_train = normalize(ds_train)
ds_eval  = normalize(ds_eval)

print("Normalized train columns:", ds_train.column_names)
print("Normalized eval  columns:", ds_eval.column_names)

columns: ['input_features', 'labels']
Detected preprocessed Whisper dataset (input_features/labels). No normalization needed.
columns: ['input_features', 'labels']
Detected preprocessed Whisper dataset (input_features/labels). No normalization needed.
Normalized train columns: ['input_features', 'labels']
Normalized eval  columns: ['input_features', 'labels']


# 3) Load Whisper + Processor
We use Whisper `small` by default.
- Language: Korean
- Task: transcribe

In [6]:
import torch
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

MODEL_ID = "openai/whisper-small"
LANGUAGE = "Korean"
TASK = "transcribe"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
processor = WhisperProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.to(device)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.config.suppress_tokens = []

print("Loaded:", MODEL_ID)

device: cuda
Loaded: openai/whisper-small


# 4) Preprocess dataset (audio → log-mel features, text → labels)
This step may take a few minutes depending on dataset size.

In [7]:
from datasets import DatasetDict

def is_preprocessed(ds):
    cols = set(ds.column_names)
    return ("input_features" in cols) and ("labels" in cols)

def prepare_batch(batch):
    audio = batch["audio"]
    if isinstance(audio, dict) and "array" in audio and "sampling_rate" in audio:
        arr = audio["array"]
        sr  = audio["sampling_rate"]
    else:
        arr = audio
        sr  = 16000

    inputs = feature_extractor(arr, sampling_rate=sr)
    batch["input_features"] = inputs.input_features[0]
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch

if is_preprocessed(ds_train) and is_preprocessed(ds_eval):
    print("Dataset already has input_features/labels. Skip prepare_batch().")
    ds_train_p = ds_train
    ds_eval_p  = ds_eval
else:
    for need in ("audio", "text"):
        if need not in ds_train.column_names or need not in ds_eval.column_names:
            raise ValueError(
                f"Raw dataset expected columns ('audio','text') "
                f"but got train={ds_train.column_names}, eval={ds_eval.column_names}"
            )

    remove_cols_train = [c for c in ds_train.column_names if c not in ("audio", "text")]
    remove_cols_eval  = [c for c in ds_eval.column_names  if c not in ("audio", "text")]

    ds_train_p = ds_train.map(
        prepare_batch,
        remove_columns=remove_cols_train,
        num_proc=1,
        desc="Preparing train"
    )
    ds_eval_p = ds_eval.map(
        prepare_batch,
        remove_columns=remove_cols_eval,
        num_proc=1,
        desc="Preparing eval"
    )

print(ds_train_p)
print(ds_eval_p)
print("train_p columns:", ds_train_p.column_names)
print("eval_p  columns:", ds_eval_p.column_names)

Dataset already has input_features/labels. Skip prepare_batch().
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 21148
})
Dataset({
    features: ['input_features', 'labels'],
    num_rows: 1114
})
train_p columns: ['input_features', 'labels']
eval_p  columns: ['input_features', 'labels']


# 5) Data collator + metrics (WER/CER via jiwer)

In [8]:
import numpy as np
from dataclasses import dataclass
from jiwer import wer as jiwer_wer, cer as jiwer_cer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

@dataclass
class WhisperDataCollator:
    processor: any

    def __call__(self, features):
        # input_features padding
        in_feats = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(in_feats, return_tensors="pt")

        # labels padding
        label_feats = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels

        # Whisper 입력엔 input_ids가 없음 (절대 넣지 않음)
        return batch

data_collator = WhisperDataCollator(processor)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = ["" if s is None else str(s) for s in pred_str]
    label_str = ["" if s is None else str(s) for s in label_str]

    return {
        "wer": float(jiwer_wer(label_str, pred_str)),
        "cer": float(jiwer_cer(label_str, pred_str)),
    }

print("Collator/metrics ready.")

Collator/metrics ready.


# 6) Baseline evaluation (before fine-tuning)
We run generation on the eval split and compute WER/CER.

In [9]:
import torch
from tqdm.auto import tqdm
from jiwer import wer as jiwer_wer, cer as jiwer_cer

@torch.inference_mode()
def evaluate_model(model, ds_eval_p, max_eval_samples=None):
    model.eval()
    preds = []
    refs = []

    it = ds_eval_p
    if max_eval_samples is not None:
        it = it.select(range(min(max_eval_samples, len(ds_eval_p))))

    for ex in tqdm(it, total=len(it)):
        input_features = torch.tensor(ex["input_features"]).unsqueeze(0).to(device)

        generated_ids = model.generate(
            input_features=input_features,
            max_new_tokens=128,
        )

        pred_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        ref_text  = tokenizer.decode(ex["labels"], skip_special_tokens=True)

        preds.append(pred_text)
        refs.append(ref_text)

    return {
        "wer": float(jiwer_wer(refs, preds)),
        "cer": float(jiwer_cer(refs, preds)),
        "n": len(preds),
    }

baseline = evaluate_model(model, ds_eval_p, max_eval_samples=50)
baseline

  0%|          | 0/50 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


{'wer': 0.4450784593437946, 'cer': 0.15777439024390244, 'n': 50}

# 7) Attach LoRA (PEFT) and fine-tune
This is designed for **classroom** runtime. Adjust steps/epochs as needed.

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

target_modules = ["q_proj", "v_proj"]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=target_modules,
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 1,769,472 || all params: 243,504,384 || trainable%: 0.7267


## Training arguments
- Use small batch + gradient accumulation
- `predict_with_generate=True` so we can compute WER/CER via generation

In [11]:
import torch
import types

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=50,
    max_steps=300,
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=128,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=lora_model,
    train_dataset=ds_train_p,
    eval_dataset=ds_eval_p,
    data_collator=data_collator,
    processing_class=processor,     # transformers 5.x 대비
    compute_metrics=compute_metrics,
)

# PEFT가 Whisper base_model에 input_ids를 넘기는 걸 막는 패치
def patched_forward(
    self,
    input_ids=None,
    attention_mask=None,
    inputs_embeds=None,
    decoder_input_ids=None,
    decoder_attention_mask=None,
    decoder_inputs_embeds=None,
    labels=None,
    **kwargs
):
    # Whisper는 input_features로 받음 -> input_ids/attention_mask/inputs_embeds를 base_model에 넘기지 않음
    return self.base_model(
        decoder_input_ids=decoder_input_ids,
        decoder_attention_mask=decoder_attention_mask,
        decoder_inputs_embeds=decoder_inputs_embeds,
        labels=labels,
        **kwargs,  # input_features가 여기 들어있음
    )

lora_model.forward = types.MethodType(patched_forward, lora_model)

trainer.train()

Step,Training Loss,Validation Loss,Wer,Cer
100,0.799800,0.785433,0.403533,0.218298
200,0.696800,0.699503,0.298358,0.089573
300,0.693400,0.671210,0.287722,0.080902


TrainOutput(global_step=300, training_loss=1.0860091336568196, metrics={'train_runtime': 2377.0123, 'train_samples_per_second': 2.019, 'train_steps_per_second': 0.126, 'total_flos': 1.397440512e+18, 'train_loss': 1.0860091336568196, 'epoch': 0.22697181766597313})

# 8) Save LoRA adapter
This saves only adapter weights (small) and training artifacts.

In [12]:
from pathlib import Path

adapter_dir = OUT_DIR / "lora_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

lora_model.save_pretrained(str(adapter_dir))
processor.save_pretrained(str(OUT_DIR / "processor"))

print("Saved adapter to:", adapter_dir)
print("Saved processor to:", OUT_DIR / "processor")

Saved adapter to: /home/data/bootcamp/user2/outputs_asr/whisper_lora_zeroth/lora_adapter
Saved processor to: /home/data/bootcamp/user2/outputs_asr/whisper_lora_zeroth/processor


# 9) Evaluate after fine-tuning
We compare with the baseline metrics.

In [13]:
after = evaluate_model(lora_model, ds_eval_p, max_eval_samples=50)
print("Baseline:", baseline)
print("After   :", after)

  0%|          | 0/50 [00:00<?, ?it/s]

Baseline: {'wer': 0.4450784593437946, 'cer': 0.15777439024390244, 'n': 50}
After   : {'wer': 0.28530670470756064, 'cer': 0.08079268292682927, 'n': 50}


# 10) Quick qualitative check (a few samples)

In [14]:
import random

def show_samples(model, ds_eval_p, k=5, seed=7):
    random.seed(seed)
    idxs = random.sample(range(len(ds_eval_p)), k=min(k, len(ds_eval_p)))
    for i in idxs:
        ex = ds_eval_p[i]
        input_features = torch.tensor(ex["input_features"]).unsqueeze(0).to(device)
        gen_ids = model.generate(input_features=input_features, max_new_tokens=128)
        hyp = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]
        ref = tokenizer.decode(ex["labels"], skip_special_tokens=True)

        print("="*80)
        print("idx:", i)
        print("REF:", ref)
        print("HYP:", hyp)

show_samples(lora_model, ds_eval_p, k=5)

idx: 663
REF: 지시한 적도 없고 요구한 적도 없다며 선관위에서 낸 보도자료만 봤는데 거기엔 특정하게 이름이 명기돼 있지 않다
HYP: 지시한 적도 없고 요구한 적도 없다며 선관위에서 낸 보도자료만 받는데 거기엔 특정하게 이름이 명기되어 있지 않다
idx: 308
REF: 이러한 증상이 있는 아이들은 간이 부으며 냉담하고 슬퍼한다
HYP:  이러한 증상이 있는 아이들은 간이 부으며 냉담하고 슬퍼한다
idx: 808
REF: 앵커 동료 교사들보다 업무량이 많았던 한 초등학교 육 학년 부장교사가 급성심장사로 숨졌습니다
HYP:  앵커 동료 교사들보다 업무량이 많았던 한 초등학교 육학년 부장교사가 급성 심장사로 숨졌습니다
idx: 98
REF: 직장과 육아를 함께하는 직장맘 이모씨는 출산한지 한 달 만에 다시 직장으로 향해야 했다
HYP: 직장과 육아를 함께하는 직장만 이모씨는 출산한 지 한 달 만에 다시 직장으로 향해야했다
idx: 148
REF: 이 환자는 최근 완치 판정을 받고 심리 치료 상담을 받기 위해 병원에 갔다가 환자와 방문자들의 항의를 받았다
HYP: 이 환자는 최근 완치 판정을 받고 심리치로 상당을 받기 위해 병원에 갔다가 환자와 방문자들의 황의를 받았다


In [15]:
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm
from jiwer import wer as jiwer_wer, cer as jiwer_cer
import time

def get_file_from_ds_eval(ex, fallback_idx=None):
    from pathlib import Path
    if isinstance(ex, dict) and "file" in ex:
        return str(ex["file"])
    if isinstance(ex, dict) and "audio" in ex:
        a = ex["audio"]
        if isinstance(a, dict) and "path" in a and a["path"]:
            return Path(a["path"]).name
    for k in ["path", "audio_path", "file_path", "wav_path"]:
        if isinstance(ex, dict) and k in ex and ex[k]:
            return Path(str(ex[k])).name
    return f"idx_{fallback_idx}" if fallback_idx is not None else "unknown"

@torch.inference_mode()
def export_tuned_csv_like_baseline(
    model,
    ds_eval,
    ds_eval_p,
    out_dir,
    model_id="openai/whisper-small+lora",   # <- 이름만 바꾸면 파일명이 바뀜
    max_new_tokens=128
):
    assert len(ds_eval) == len(ds_eval_p), "ds_eval and ds_eval_p length mismatch"

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    safe_id = model_id.replace("/", "_")      # 튜닝 전 네이밍과 동일 규칙
    hyp_path  = out_dir / f"hyp_{safe_id}.csv"
    eval_path = out_dir / f"eval_{safe_id}.csv"

    model.eval()
    rows = []
    preds, refs = [], []

    t0 = time.time()

    for i in tqdm(range(len(ds_eval_p)), desc=f"EXPORT: {model_id}"):
        ex_p = ds_eval_p[i]
        ex   = ds_eval[i]

        file_name = get_file_from_ds_eval(ex, fallback_idx=i)

        input_features = torch.tensor(ex_p["input_features"]).unsqueeze(0).to(device)
        gen_ids = model.generate(input_features=input_features, max_new_tokens=max_new_tokens)

        hyp = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]
        ref = tokenizer.decode(ex_p["labels"], skip_special_tokens=True)

        rows.append({"file": file_name, "ref": ref, "hyp": hyp, "model_id": safe_id})
        preds.append(hyp); refs.append(ref)

    elapsed = time.time() - t0
    w = float(jiwer_wer(refs, preds))
    c = float(jiwer_cer(refs, preds))

    df_eval = pd.DataFrame(rows)
    df_hyp = df_eval[["file", "hyp", "model_id"]].copy()

    df_hyp.to_csv(hyp_path, index=False, encoding="utf-8-sig")
    df_eval.to_csv(eval_path, index=False, encoding="utf-8-sig")

    print("Saved hyp:", hyp_path)
    print("Saved eval:", eval_path)
    print("WER:", w, "| CER:", c, "| elapsed(s):", round(elapsed, 3))

    return df_hyp, df_eval, elapsed, w, c

In [16]:
# 튜닝 전 결과랑 같은 폴더에 저장하고 싶으면 OUT_DIR를 같은 곳으로
OUT_SAVE_DIR = Path.home() / "outputs_asr"  # 예: Path("/content/bootcamp/outputs_asr")

df_hyp_lora, df_eval_lora, elapsed, w, c = export_tuned_csv_like_baseline(
    model=lora_model,
    ds_eval=ds_eval,
    ds_eval_p=ds_eval_p,
    out_dir=OUT_SAVE_DIR,
    model_id="openai/whisper-small+lora",   # <- 파일명이 hyp_openai_whisper-small+lora.csv 처럼 됨
)

EXPORT: openai/whisper-small+lora:   0%|          | 0/1114 [00:00<?, ?it/s]

Saved hyp: /home/data/bootcamp/user2/outputs_asr/hyp_openai_whisper-small+lora.csv
Saved eval: /home/data/bootcamp/user2/outputs_asr/eval_openai_whisper-small+lora.csv
WER: 0.287722353526558 | CER: 0.08090157678504734 | elapsed(s): 805.067


# Notes / Troubleshooting
- If you run out of GPU memory:
  - lower `per_device_train_batch_size`
  - increase `gradient_accumulation_steps`
  - reduce `max_steps`
- If dataset columns differ:
  - check `ds_train.column_names` and adjust the `normalize()` function
- For recorded data:
  - ensure `refs.csv` rows have transcript filled and corresponding `.wav` files exist